# Fine-tune Cross-Encoder v0.6 - Ensemble 5 Runs

**Structure**: 1 notebook, 7 cells độc lập
- Cell 1-2: Setup (chạy 1 lần)
- Cell 3-7: Run 1-5 (mỗi cell train 1 model riêng, có thể chạy lúc khác nhau)
- Cell 8: Aggregate (chạy sau khi tất cả 5 runs hoàn thành)

**Lợi ích**:
- ✅ 1 notebook dễ quản lý
- ✅ Mỗi run ~60 phút, không vượt GPU limit Colab
- ✅ Nếu ngắt GPU, chạy lại cell đó mà không ảnh hưởng cells khác
- ✅ Models tự động save, upload Drive

| | |
|---|---|
| **Seeds** | [42, 123, 456, 789, 999] |
| **LR** | 5e-5 (optimal from Phase 2.1) |
| **BS** | 16 |
| **Epochs** | 15 |
| **Expected** | +0.1-0.5pp over baseline 65.15% |

In [1]:
# === SETUP CELL 1: Clone & Install ===
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.6
!git pull origin experiment/cross-encoder-v0.6
!pip install -q -r requirements.txt

import torch
print("✅ CUDA available:", torch.cuda.is_available())
print("✅ Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Cloning into '/content/Ai-Recruiter-Mini-Ai-Service'...
remote: Enumerating objects: 7406, done.
remote: Counting objects: 100% (387/387), done.
remote: Compressing objects: 100% (216/216), done.
remote: Total 7406 (delta 225), reused 204 (delta 170), pack-reused 7019 (from 2)
Receiving objects: 100% (7406/7406), 32.36 MiB | 13.21 MiB/s, done.
Resolving deltas: 100% (4402/4402), done.
/content/Ai-Recruiter-Mini-Ai-Service
Branch 'experiment/cross-encoder-v0.6' set up to track remote branch 'experiment/cross-encoder-v0.6' from 'origin'.
Switched to a new branch 'experiment/cross-encoder-v0.6'
From https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service
 * branch            experiment/cross-encoder-v0.6 -> FETCH_HEAD
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━

In [2]:
# === SETUP CELL 2: Load Data & Helpers ===
import json
import torch
import numpy as np
import random
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
from typing import Any

class CECorrelationEvaluator:
    """Evaluate cross-encoder using Spearman correlation."""
    def __init__(self, sentence_pairs: list, labels_0_1: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1 = labels_0_1
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = ""):
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_1=[ex.label for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0

def load_jsonl(path: str):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def load_dataset(data_dir: str):
    train_data = load_jsonl(f"{data_dir}/cross_encoder_train.jsonl")
    val_data = load_jsonl(f"{data_dir}/cross_encoder_validation.jsonl")
    test_data = load_jsonl(f"{data_dir}/cross_encoder_test.jsonl")

    def to_input_examples(records):
        return [
            InputExample(texts=[rec['cv_text'], rec['jd_text']], label=rec['label'])
            for rec in records
        ]

    return to_input_examples(train_data), to_input_examples(val_data), to_input_examples(test_data)

def compute_metrics(model: CrossEncoder, examples: list[InputExample]):
    preds = model.predict([ex.texts for ex in examples], batch_size=32, show_progress_bar=False)
    preds_100 = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])

    mae = np.mean(np.abs(preds_100 - labels_100))
    rmse = np.sqrt(np.mean((preds_100 - labels_100) ** 2))
    label_acc = np.mean(np.abs(preds_100 - labels_100) <= 10)

    return {'MAE': mae, 'RMSE': rmse, 'LabelAcc': label_acc}

# Load dataset (once)
train_examples, val_examples, test_examples = load_dataset('datasets/versions/v0.5/cross_encoder')

print(f"✅ Data loaded: {len(train_examples)} train, {len(val_examples)} val, {len(test_examples)} test")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ Data loaded: 9350 train, 2000 val, 2000 test


## RUN 1 (seed=42) - Chạy cell này độc lập

In [ ]:
# === RUN 1 CELL: seed=42 ===
import torch
from torch.utils.data import DataLoader
import os

# Config
seed = 42
run_idx = 1
base_model = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
learning_rate = 5e-5
epochs = 15
batch_size = 16

# Set seed
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

total_steps = (len(train_examples) // batch_size + 1) * epochs
warmup_steps = int(total_steps * 0.1)

run_name = f"v0.6-ensemble-run{run_idx}-seed{seed}"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

print(f"\n🚀 Run {run_idx}/5: seed={seed}")
print(f"Config: LR={learning_rate:.0e}, BS={batch_size}, epochs={epochs}")

# Initialize & train
model = CrossEncoder(base_model, num_labels=1, default_activation_function=torch.nn.Sigmoid())
evaluator = CECorrelationEvaluator.from_input_examples(val_examples)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=epochs,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': learning_rate},
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

# Evaluate
best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ Run {run_idx} completed (seed={seed})")
print(f"  Val LabelAcc:  {val_metrics['LabelAcc']*100:.2f}%")
print(f"  Test LabelAcc: {test_metrics['LabelAcc']*100:.2f}%")

# Save report
os.makedirs('artifacts/reports', exist_ok=True)
report = {
    "run": run_idx,
    "seed": seed,
    "run_name": run_name,
    "val_labelacc": float(val_metrics['LabelAcc']),
    "test_labelacc": float(test_metrics['LabelAcc']),
    "val_mae": float(val_metrics['MAE']),
    "test_mae": float(test_metrics['MAE']),
    "val_rmse": float(val_metrics['RMSE']),
    "test_rmse": float(test_metrics['RMSE'])
}

report_path = f'artifacts/reports/fine_tune_cross_encoder_v0.6_ensemble_run{run_idx}_seed{seed}_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"✅ Report saved: {report_path}")

# Upload to Drive
try:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive', force_remount=True)
    drive_base = "/content/drive/MyDrive/ai-recruiter"
    os.makedirs(f"{drive_base}/models", exist_ok=True)
    os.makedirs(f"{drive_base}/reports", exist_ok=True)

    dest_model = f"{drive_base}/models/{run_name}"
    if os.path.exists(dest_model):
        shutil.rmtree(dest_model)
    shutil.copytree(output_dir, dest_model)

    shutil.copy(report_path, f"{drive_base}/reports/fine_tune_cross_encoder_v0.6_ensemble_run{run_idx}_seed{seed}_report.json")
    print(f"✅ Model & report uploaded to Drive")
except Exception as e:
    print(f"⚠️ Drive upload skipped or failed: {e}")


🚀 Run 1/5: seed=42
Config: LR=5e-05, BS=16, epochs=15


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]


✅ Run 1 completed (seed=42)
  Val LabelAcc:  64.55%
  Test LabelAcc: 62.55%
✅ Report saved: artifacts/reports/fine_tune_cross_encoder_v0.6_ensemble_run1_seed42_report.json
Mounted at /content/drive
✅ Model & report uploaded to Drive


## RUN 2 (seed=123) - Chạy cell này độc lập

In [ ]:
# === RUN 2 CELL: seed=123 ===
# (Identical structure to Run 1, just change seed & run_idx)

seed = 123
run_idx = 2
base_model = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
learning_rate = 5e-5
epochs = 15
batch_size = 16

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

total_steps = (len(train_examples) // batch_size + 1) * epochs
warmup_steps = int(total_steps * 0.1)
run_name = f"v0.6-ensemble-run{run_idx}-seed{seed}"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

print(f"\n🚀 Run {run_idx}/5: seed={seed}")
model = CrossEncoder(base_model, num_labels=1, default_activation_function=torch.nn.Sigmoid())
evaluator = CECorrelationEvaluator.from_input_examples(val_examples)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=epochs,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': learning_rate},
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ Run {run_idx} completed (seed={seed})")
print(f"  Val LabelAcc:  {val_metrics['LabelAcc']*100:.2f}%")
print(f"  Test LabelAcc: {test_metrics['LabelAcc']*100:.2f}%")

os.makedirs('artifacts/reports', exist_ok=True)
report = {
    "run": run_idx,
    "seed": seed,
    "run_name": run_name,
    "val_labelacc": float(val_metrics['LabelAcc']),
    "test_labelacc": float(test_metrics['LabelAcc']),
    "val_mae": float(val_metrics['MAE']),
    "test_mae": float(test_metrics['MAE']),
    "val_rmse": float(val_metrics['RMSE']),
    "test_rmse": float(test_metrics['RMSE'])
}

report_path = f'artifacts/reports/fine_tune_cross_encoder_v0.6_ensemble_run{run_idx}_seed{seed}_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"✅ Report saved")

try:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive', force_remount=True)
    drive_base = "/content/drive/MyDrive/ai-recruiter"
    dest_model = f"{drive_base}/models/{run_name}"
    if os.path.exists(dest_model):
        shutil.rmtree(dest_model)
    shutil.copytree(output_dir, dest_model)
    shutil.copy(report_path, f"{drive_base}/reports/fine_tune_cross_encoder_v0.6_ensemble_run{run_idx}_seed{seed}_report.json")
    print(f"✅ Uploaded to Drive")
except:
    print(f"⚠️ Drive upload skipped")


🚀 Run 2/5: seed=123


Epoch:   0%|          | 0/15 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]


✅ Run 2 completed (seed=123)
  Val LabelAcc:  66.05%
  Test LabelAcc: 64.95%
✅ Report saved
Mounted at /content/drive
✅ Uploaded to Drive


## RUN 3 (seed=456) - Chạy cell này độc lập

In [ ]:
# === RUN 3 CELL: seed=456 ===
seed = 456
run_idx = 3
base_model = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
learning_rate = 5e-5
epochs = 15
batch_size = 16

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

total_steps = (len(train_examples) // batch_size + 1) * epochs
warmup_steps = int(total_steps * 0.1)
run_name = f"v0.6-ensemble-run{run_idx}-seed{seed}"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

print(f"\n🚀 Run {run_idx}/5: seed={seed}")
model = CrossEncoder(base_model, num_labels=1, default_activation_function=torch.nn.Sigmoid())
evaluator = CECorrelationEvaluator.from_input_examples(val_examples)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=epochs,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': learning_rate},
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ Run {run_idx} completed (seed={seed})")
print(f"  Val LabelAcc:  {val_metrics['LabelAcc']*100:.2f}%")
print(f"  Test LabelAcc: {test_metrics['LabelAcc']*100:.2f}%")

os.makedirs('artifacts/reports', exist_ok=True)
report = {
    "run": run_idx,
    "seed": seed,
    "run_name": run_name,
    "val_labelacc": float(val_metrics['LabelAcc']),
    "test_labelacc": float(test_metrics['LabelAcc']),
    "val_mae": float(val_metrics['MAE']),
    "test_mae": float(test_metrics['MAE']),
    "val_rmse": float(val_metrics['RMSE']),
    "test_rmse": float(test_metrics['RMSE'])
}

report_path = f'artifacts/reports/fine_tune_cross_encoder_v0.6_ensemble_run{run_idx}_seed{seed}_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"✅ Report saved")

try:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive', force_remount=True)
    drive_base = "/content/drive/MyDrive/ai-recruiter"
    dest_model = f"{drive_base}/models/{run_name}"
    if os.path.exists(dest_model):
        shutil.rmtree(dest_model)
    shutil.copytree(output_dir, dest_model)
    shutil.copy(report_path, f"{drive_base}/reports/fine_tune_cross_encoder_v0.6_ensemble_run{run_idx}_seed{seed}_report.json")
    print(f"✅ Uploaded to Drive")
except:
    print(f"⚠️ Drive upload skipped")


🚀 Run 3/5: seed=456


Epoch:   0%|          | 0/15 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]


✅ Run 3 completed (seed=456)
  Val LabelAcc:  64.05%
  Test LabelAcc: 63.60%
✅ Report saved
Mounted at /content/drive
✅ Uploaded to Drive


## RUN 4 (seed=789) - Chạy cell này độc lập

In [ ]:
# === RUN 4 CELL: seed=789 ===
seed = 789
run_idx = 4
base_model = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
learning_rate = 5e-5
epochs = 15
batch_size = 16

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

total_steps = (len(train_examples) // batch_size + 1) * epochs
warmup_steps = int(total_steps * 0.1)
run_name = f"v0.6-ensemble-run{run_idx}-seed{seed}"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

print(f"\n🚀 Run {run_idx}/5: seed={seed}")
model = CrossEncoder(base_model, num_labels=1, default_activation_function=torch.nn.Sigmoid())
evaluator = CECorrelationEvaluator.from_input_examples(val_examples)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=epochs,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': learning_rate},
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ Run {run_idx} completed (seed={seed})")
print(f"  Val LabelAcc:  {val_metrics['LabelAcc']*100:.2f}%")
print(f"  Test LabelAcc: {test_metrics['LabelAcc']*100:.2f}%")

os.makedirs('artifacts/reports', exist_ok=True)
report = {
    "run": run_idx,
    "seed": seed,
    "run_name": run_name,
    "val_labelacc": float(val_metrics['LabelAcc']),
    "test_labelacc": float(test_metrics['LabelAcc']),
    "val_mae": float(val_metrics['MAE']),
    "test_mae": float(test_metrics['MAE']),
    "val_rmse": float(val_metrics['RMSE']),
    "test_rmse": float(test_metrics['RMSE'])
}

report_path = f'artifacts/reports/fine_tune_cross_encoder_v0.6_ensemble_run{run_idx}_seed{seed}_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"✅ Report saved")

try:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive', force_remount=True)
    drive_base = "/content/drive/MyDrive/ai-recruiter"
    dest_model = f"{drive_base}/models/{run_name}"
    if os.path.exists(dest_model):
        shutil.rmtree(dest_model)
    shutil.copytree(output_dir, dest_model)
    shutil.copy(report_path, f"{drive_base}/reports/fine_tune_cross_encoder_v0.6_ensemble_run{run_idx}_seed{seed}_report.json")
    print(f"✅ Uploaded to Drive")
except:
    print(f"⚠️ Drive upload skipped")


🚀 Run 4/5: seed=789


Epoch:   0%|          | 0/15 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]


✅ Run 4 completed (seed=789)
  Val LabelAcc:  64.75%
  Test LabelAcc: 63.70%
✅ Report saved
Mounted at /content/drive
✅ Uploaded to Drive


## RUN 5 (seed=999) - Chạy cell này độc lập

In [4]:
# === RUN 5 CELL: seed=999 ===
seed = 999
run_idx = 5
base_model = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
learning_rate = 5e-5
epochs = 15
batch_size = 16

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

total_steps = (len(train_examples) // batch_size + 1) * epochs
warmup_steps = int(total_steps * 0.1)
run_name = f"v0.6-ensemble-run{run_idx}-seed{seed}"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

print(f"\n🚀 Run {run_idx}/5: seed={seed}")
model = CrossEncoder(base_model, num_labels=1, default_activation_function=torch.nn.Sigmoid())
evaluator = CECorrelationEvaluator.from_input_examples(val_examples)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=epochs,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': learning_rate},
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ Run {run_idx} completed (seed={seed})")
print(f"  Val LabelAcc:  {val_metrics['LabelAcc']*100:.2f}%")
print(f"  Test LabelAcc: {test_metrics['LabelAcc']*100:.2f}%")

os.makedirs('artifacts/reports', exist_ok=True)
report = {
    "run": run_idx,
    "seed": seed,
    "run_name": run_name,
    "val_labelacc": float(val_metrics['LabelAcc']),
    "test_labelacc": float(test_metrics['LabelAcc']),
    "val_mae": float(val_metrics['MAE']),
    "test_mae": float(test_metrics['MAE']),
    "val_rmse": float(val_metrics['RMSE']),
    "test_rmse": float(test_metrics['RMSE'])
}

report_path = f'artifacts/reports/fine_tune_cross_encoder_v0.6_ensemble_run{run_idx}_seed{seed}_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"✅ Report saved")

try:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive', force_remount=True)
    drive_base = "/content/drive/MyDrive/ai-recruiter"
    dest_model = f"{drive_base}/models/{run_name}"
    if os.path.exists(dest_model):
        shutil.rmtree(dest_model)
    shutil.copytree(output_dir, dest_model)
    shutil.copy(report_path, f"{drive_base}/reports/fine_tune_cross_encoder_v0.6_ensemble_run{run_idx}_seed{seed}_report.json")
    print(f"✅ Uploaded to Drive")
except:
    print(f"⚠️ Drive upload skipped")


🚀 Run 5/5: seed=999


Epoch:   0%|          | 0/15 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]


✅ Run 5 completed (seed=999)
  Val LabelAcc:  65.30%
  Test LabelAcc: 64.70%
✅ Report saved
Mounted at /content/drive
⚠️ Drive upload skipped


## AGGREGATE CELL - Chạy sau khi tất cả 5 runs hoàn thành

In [5]:
# === AGGREGATE CELL: Load 5 models, ensemble predictions ===
from google.colab import drive
import shutil

# Mount Drive & load 5 models
drive.mount('/content/drive', force_remount=True)
drive_base = "/content/drive/MyDrive/ai-recruiter/models"

seeds = [42, 123, 456, 789, 999]
all_predictions = []
all_results = []

print("📊 Loading 5 models from Drive...\n")

for run_idx, seed in enumerate(seeds, 1):
    run_name = f"v0.6-ensemble-run{run_idx}-seed{seed}"
    drive_model_path = f"{drive_base}/{run_name}"

    # Check if model exists on Drive
    if not os.path.exists(drive_model_path):
        print(f"❌ Run {run_idx} (seed={seed}): Model NOT found on Drive")
        continue

    print(f"✅ Run {run_idx} (seed={seed}): Loading model...")
    model = CrossEncoder(drive_model_path)

    # Get predictions on test set
    preds = model.predict([ex.texts for ex in test_examples], batch_size=32, show_progress_bar=False)
    all_predictions.append(preds)

    # Compute metrics
    metrics = compute_metrics(model, test_examples)
    result = {
        "run": run_idx,
        "seed": seed,
        "test_labelacc": float(metrics['LabelAcc']),
        "test_mae": float(metrics['MAE']),
        "test_rmse": float(metrics['RMSE'])
    }
    all_results.append(result)
    print(f"   Test LabelAcc: {metrics['LabelAcc']*100:.2f}%")

if len(all_predictions) < 5:
    print(f"\n❌ ERROR: Not all 5 models found! Found {len(all_predictions)}/5")
else:
    print(f"\n✅ All 5 models loaded!")

    # Ensemble: average predictions
    ensemble_preds = np.mean(all_predictions, axis=0)
    ensemble_preds_100 = ensemble_preds * 100
    test_labels_100 = np.array([ex.label * 100 for ex in test_examples])

    # Compute ensemble metrics
    ensemble_mae = np.mean(np.abs(ensemble_preds_100 - test_labels_100))
    ensemble_rmse = np.sqrt(np.mean((ensemble_preds_100 - test_labels_100) ** 2))
    ensemble_label_acc = np.mean(np.abs(ensemble_preds_100 - test_labels_100) <= 10)

    print(f"\n📊 RESULTS:")
    print(f"{'Run':<6} {'Seed':<8} {'Test Acc':<12}")
    print(f"{'-'*26}")
    for result in all_results:
        print(f"{result['run']:<6} {result['seed']:<8} {result['test_labelacc']*100:>10.2f}%")

    test_accs = [r['test_labelacc'] for r in all_results]
    print(f"\n📈 Ensemble Summary:")
    print(f"  Best single run:     {np.max(test_accs)*100:.2f}%")
    print(f"  Worst single run:    {np.min(test_accs)*100:.2f}%")
    print(f"  Mean:                {np.mean(test_accs)*100:.2f}%")
    print(f"  Std dev:             {np.std(test_accs)*100:.4f}pp")
    print(f"\n  Ensemble Test Acc:   {ensemble_label_acc*100:.2f}%")
    print(f"  Ensemble MAE:        {ensemble_mae:.4f}")
    print(f"  Ensemble RMSE:       {ensemble_rmse:.4f}")

    baseline = 0.6515
    improvement = (ensemble_label_acc - baseline) * 100
    print(f"\n  Baseline (Phase 2.1):  {baseline*100:.2f}%")
    print(f"  Improvement:          {improvement:+.2f}pp")

    # Save ensemble report
    os.makedirs('artifacts/reports', exist_ok=True)
    ensemble_report = {
        "experiment": "Phase 5: Ensemble 5 Runs with Different Seeds",
        "individual_runs": all_results,
        "ensemble_metrics": {
            "test_labelacc": float(ensemble_label_acc),
            "test_mae": float(ensemble_mae),
            "test_rmse": float(ensemble_rmse)
        },
        "statistics": {
            "best_single_run_test_labelacc": float(np.max(test_accs)),
            "worst_single_run_test_labelacc": float(np.min(test_accs)),
            "mean_test_labelacc": float(np.mean(test_accs)),
            "std_dev_test_labelacc": float(np.std(test_accs))
        },
        "comparison_to_baseline": {
            "baseline_test_labelacc": 0.6515,
            "ensemble_improvement_pp": float(improvement)
        }
    }

    report_path = 'artifacts/reports/fine_tune_cross_encoder_v0.6_ensemble_5runs_aggregate_report.json'
    with open(report_path, 'w') as f:
        json.dump(ensemble_report, f, indent=2)
    print(f"\n✅ Ensemble report saved: {report_path}")

Mounted at /content/drive
📊 Loading 5 models from Drive...

✅ Run 1 (seed=42): Loading model...
   Test LabelAcc: 62.55%
✅ Run 2 (seed=123): Loading model...
   Test LabelAcc: 64.95%
✅ Run 3 (seed=456): Loading model...
   Test LabelAcc: 63.60%
✅ Run 4 (seed=789): Loading model...
   Test LabelAcc: 63.70%
✅ Run 5 (seed=999): Loading model...
   Test LabelAcc: 64.70%

✅ All 5 models loaded!

📊 RESULTS:
Run    Seed     Test Acc    
--------------------------
1      42            62.55%
2      123           64.95%
3      456           63.60%
4      789           63.70%
5      999           64.70%

📈 Ensemble Summary:
  Best single run:     64.95%
  Worst single run:    62.55%
  Mean:                63.90%
  Std dev:             0.8597pp

  Ensemble Test Acc:   65.80%
  Ensemble MAE:        9.1147
  Ensemble RMSE:       12.8660

  Baseline (Phase 2.1):  65.15%
  Improvement:          +0.65pp

✅ Ensemble report saved: artifacts/reports/fine_tune_cross_encoder_v0.6_ensemble_5runs_aggregate_r